In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용

In [2]:
from pathlib import Path

BENCH_ID = "AIHub_KsponSpeech_general_clean"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_KsponSpeech_Clean/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인 후 지정

OUT_DIR = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
from eval_silver import convert_silver

conv = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv, corpus_id=BENCH_ID)
print(f"{n} samples → {conv}")

3000 samples → ../BENCHMARK/results/whisper_small__AIHub_KsponSpeech_general_clean/_silver_converted/AIHub_KsponSpeech_general_clean.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=26.153281994866152, scer=26.853885887312234, wer=46.92918725176789, samples=3000, per_sample_cer=[2.898550724637681, 12.5, 5.88235294117647, 9.30232558139535, 69.23076923076923, 0.0, 3.125, 25.0, 26.923076923076923, 30.303030303030305, 16.216216216216218, 0.0, 14.285714285714285, 20.0, 50.0, 75.0, 13.043478260869565, 20.588235294117645, 15.384615384615385, 11.666666666666666, 11.11111111111111, 22.641509433962266, 38.095238095238095, 5.263157894736842, 3.3333333333333335, 33.33333333333333, 24.324324324324326, 0.0, 11.538461538461538, 84.0, 0.0, 6.25, 0.0, 8.333333333333332, 7.142857142857142, 48.0, 4.3478260869565215, 0.0, 32.91139240506329, 6.25, 22.22222222222222, 30.76923076923077, 22.58064516129032, 20.0, 13.333333333333334, 52.38095238095239, 25.0, 6.666666666666667, 7.6923076923076925, 27.27272727272727, 13.636363636363635, 34.146341463414636, 9.090909090909092, 0.0, 26.08695652173913, 14.285714285714285, 51.61290322580645, 7.6923076923076925, 14.285714285714285, 1

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_KsponSpeech_general_clean
   Date: 2026-06-12T15:30:22

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_KsponSpeech_general_clean                              26.15      26.85      3,000
--------------------------------------------------------------------------------
Weighted Average                                             26.15                 3,000

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_KsponSpeech_general_clean
  [by age_group]
  value                   CER (%)    samples
  unknown                   26.15      3,000
  [by gender]
  value                   CER (%)    samples
  unknown                   26.

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

worst = df.sort_values("cer", ascending=False).head(20)
for _, row in worst.iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 19766.7] 정답: 그래서
             예측: 고맙습니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다

[CER 7966.7] 정답: 그래서
             예측: 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스 크리스마스

[CER 7462.5] 정답: 그럼 이제 그냥
             예측: 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사